# Limpieza del dataset de recetas

Perfil inicial → nulos y texto → campos numéricos → listas e ingredientes → duplicados → resultados.

In [1]:
from pathlib import Path
import ast
import hashlib
import json

import pandas as pd
from ftfy import fix_encoding
from ftfy.badness import is_bad

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'JavaProcompras/pom.xml').is_file())
ORIGEN = RAIZ / 'JavaProcompras/src/main/resources/datasets/recetasdelaabuela.csv'
PROCESADOS = RAIZ / 'notebook/data/processed'
REPORTES = RAIZ / 'notebook/data/reports'
hash_original = hashlib.sha256(ORIGEN.read_bytes()).hexdigest()

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 12)

## 1. Perfil inicial

In [2]:
raw = pd.read_csv(ORIGEN, dtype='string', encoding='utf-8-sig', keep_default_na=False)
raw = raw.replace(r'^\s*$', pd.NA, regex=True)

raw.shape, raw.head(3)

((20085, 19),   Id                                                         Nombre  \
0  1                                            Berenjenas rellenas   
1  2                          Alcachofas al horno con pico de gallo   
2  3  Arroz basmati salteado con Heura y verduras variadas al curry   

                                                         URL  \
0  https://www.elmueble.com/cocinas/comidas-saludables_46627   
1  https://www.elmueble.com/cocinas/comidas-saludables_46627   
2  https://www.elmueble.com/cocinas/comidas-saludables_46627   

                                                                                          Ingredientes  \
0  2 berenjenas, 1 pimiento rojo, 1 pimiento amarillo, 1 pimiento verde, 1 calabacín, 1 cebolleta, ...   
1  4 alcachofas, 1 limón, 400 g de carne picada (mitad ternera, mitad cerdo), 1 cebolla blanca, 1 c...   
2  200 g de arroz basmati integral, 450 ml de agua, 1 zanahoria, 1/2 pimiento rojo, 1 pimiento verd...   

                   

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 20085 entries, 0 to 20084
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  20085 non-null  string
 1   Nombre              20085 non-null  string
 2   URL                 20085 non-null  string
 3   Ingredientes        19876 non-null  string
 4   Pasos               20083 non-null  string
 5   Pais                19589 non-null  string
 6   Duracion            13252 non-null  string
 7   Porciones           18858 non-null  string
 8   Calorias            0 non-null      string
 9   Categoria           13126 non-null  string
 10  Contexto            11722 non-null  string
 11  Valoracion y Votos  13422 non-null  string
 12  Comensales          12578 non-null  string
 13  Tiempo              10558 non-null  string
 14  Dificultad          13782 non-null  string
 15  Categoria 2         9005 non-null   string
 16  Unnamed: 16         0 non-null   

In [4]:
perfil_inicial = pd.DataFrame({
    'tipo': raw.dtypes.astype(str),
    'nulos': raw.isna().sum(),
    'nulos_pct': raw.isna().mean().mul(100).round(2),
    'distintos': raw.nunique(),
}).sort_values('nulos_pct', ascending=False)

perfil_inicial

,tipo,nulos,nulos_pct,distintos
Unnamed: 16,string,20085,100.00,0
Unnamed: 17,string,20085,100.00,0
Calorias,string,20085,100.00,0
Categoria 2,string,11080,55.17,212
Tiempo,string,9527,47.43,8
Contexto,string,8363,41.64,11722
Comensales,string,7507,37.38,28
Categoria,string,6959,34.65,701
Duracion,string,6833,34.02,81
Valoracion y Votos,string,6663,33.17,1570


In [5]:
fuentes = raw['URL'].str.extract(r'^https?://([^/]+)', expand=False).rename('fuente')
perfil_fuentes = (
    raw.assign(fuente=fuentes, tiene_comensales=raw['Comensales'].notna())
       .groupby('fuente', dropna=False)
       .agg(recetas=('Id', 'size'), con_comensales=('tiene_comensales', 'sum'))
       .assign(cobertura_comensales_pct=lambda t: t['con_comensales'].div(t['recetas']).mul(100).round(1))
       .sort_values('recetas', ascending=False)
)
perfil_fuentes

,recetas,con_comensales,cobertura_comensales_pct
fuente,,,
www.recetasgratis.net,18396,12578,68.4
www.yanuq.com,832,0,0.0
cookpad.com,720,0,0.0
www.directoalpaladar.com,92,0,0.0
www.elmueble.com,37,0,0.0
www.vitonica.com,8,0,0.0


## 2. Nulos, nombres de columnas y espacios

`[]` representa una lista vacía. Eliminamos columnas completamente vacías y conservamos el original como referencia.

In [6]:
df = raw.copy()
df.columns = (df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')
                       .str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
df = df.rename(columns={'unnamed_18': 'nombre_auxiliar', 'tiempo': 'tipo_comida'})
df = df.replace('[]', pd.NA)

columnas_vacias = df.columns[df.isna().all()].tolist()
df = df.dropna(axis='columns', how='all')
df['fuente'] = fuentes
df = df.set_index('id', drop=False)

pd.Series(columnas_vacias, name='columnas_eliminadas')

0      calorias
1    unnamed_16
2    unnamed_17
Name: columnas_eliminadas, dtype: str

In [7]:
columnas_texto = df.select_dtypes(include='string').columns.difference(['id', 'url', 'fuente'])
antes_texto = df[columnas_texto].copy()

df[columnas_texto] = df[columnas_texto].apply(
    lambda s: s.str.normalize('NFC').str.replace(r'\s+', ' ', regex=True).str.strip()
)
df = df.replace(r'^\s*$', pd.NA, regex=True)

antes_texto.compare(df[columnas_texto], result_names=('antes', 'despues')).head(6)

contexto         ingredientes         nombre         nombre_auxiliar  \
      antes despues        antes despues  antes despues           antes   
id                                                                        
3      <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   
7      <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   
10     <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   
11     <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   
12     <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   
13     <NA>    <NA>         <NA>    <NA>   <NA>    <NA>            <NA>   

            \
   despues   
id           
3     <NA>   
7     <NA>   
10    <NA>   
11    <NA>   
12    <NA>   
13    <NA>   

                                                                                                  pasos  \
                                                                                                  antes   
id                                                                                                        
3   Paso 1. Hervir el arroz basmati durante unos 25 minutos, y una vez cocinado, colar y pasar por a...   
7   Paso 1. Eliminar la capa exterior del puerro y reservar la parte de color verde. Limpiar la otra...   
10  Paso 1. Lavar y cortar el tomate en forma de dados. Reservar. , Paso 2. Pelar y picar la cebolle...   
11  Paso 1. El día anterior, limpiar bien los filetes de salmón de piel y espinas. Disponer en una f...   
12  Paso 1. Para la vinagreta, poner en remojo los tomates secos al menos 4 horas antes. Luego, cuan...   
13  Paso 1. Cocer los guisantes en agua salada 5 minutos y escurrir., Paso 2. Pelar y picar la cebol...   

                                                                                                         \
                                                                                                despues   
id                                                                                                        
3   Paso 1. Hervir el arroz basmati durante unos 25 minutos, y una vez cocinado, colar y pasar por a...   
7   Paso 1. Eliminar la capa exterior del puerro y reservar la parte de color verde. Limpiar la otra...   
10  Paso 1. Lavar y cortar el tomate en forma de dados. Reservar. , Paso 2. Pelar y picar la cebolle...   
11  Paso 1. El día anterior, limpiar bien los filetes de salmón de piel y espinas. Disponer en una f...   
12  Paso 1. Para la vinagreta, poner en remojo los tomates secos al menos 4 horas antes. Luego, cuan...   
13  Paso 1. Cocer los guisantes en agua salada 5 minutos y escurrir., Paso 2. Pelar y picar la cebol...   

   porciones          
       antes despues  
id                    
3       <NA>    <NA>  
7       <NA>    <NA>  
10      <NA>    <NA>  
11      <NA>    <NA>  
12      <NA>    <NA>  
13      <NA>    <NA>

## 3. Reparar codificación con ftfy

Guardamos el antes y el después de las reparaciones. Las señales restantes se revisan al final.

In [8]:
antes_codificacion = df[columnas_texto].copy()
codificacion_antes = antes_codificacion.apply(lambda s: s.map(is_bad, na_action='ignore')).fillna(False).any(axis=1)

df[columnas_texto] = df[columnas_texto].apply(
    lambda s: s.map(fix_encoding, na_action='ignore').str.normalize('NFC')
)

cambios_codificacion = (
    antes_codificacion.compare(df[columnas_texto], result_names=('antes', 'despues'))
                      .rename_axis(columns=['campo', 'valor'])
                      .stack(level='campo').dropna(how='all').reset_index()
)
codificacion_despues = df[columnas_texto].apply(lambda s: s.map(is_bad, na_action='ignore')).fillna(False).any(axis=1)

cambios_codificacion.head(10)

valor,id,campo,antes,despues
0,19366,ingredientes,"[' 2 kilos panceta de cerdo', ' 2 cucharaditas sal', ' 1 cucharadita pimienta', ' 1/2 cucharadit...","[' 2 kilos panceta de cerdo', ' 2 cucharaditas sal', ' 1 cucharadita pimienta', ' 1/2 cucharadit..."
1,19366,pasos,"[' 1 En una licuadora ponemos el ajÃ­ panca, sal,pimienta,comino,ajos,la rama de Huacatay y el a...","[' 1 En una licuadora ponemos el ají panca, sal,pimienta,comino,ajos,la rama de Huacatay y el ag..."
2,19367,ingredientes,"[' Enharinado de la olla', ' Mantequilla ðŸ§ˆ y harina, aceite y Papel manteca', ' Masa', ' 1 ta...","[' Enharinado de la olla', ' Mantequilla 🧈 y harina, aceite y Papel manteca', ' Masa', ' 1 taza ..."
3,19367,pasos,"[' 1 Ingredientes Secos: 1taza Harina, 1taza azÃºcar, 1/2 chdta.ðŸ¥„de sal, polvo de hornear,(Op...","[' 1 Ingredientes Secos: 1taza Harina, 1taza azúcar, 1/2 chdta.🥄de sal, polvo de hornear,(Opcion..."
4,19368,ingredientes,"[' 2 tazas Arroz del dia anterior', ' 2 tazas frejoles, pueden ser del dÃ­a anterior o congelado...","[' 2 tazas Arroz del dia anterior', ' 2 tazas frejoles, pueden ser del día anterior o congelados..."
5,19368,pasos,"[' 1 El frejol se adereza friendo el tocino, una vez dorado se pone la cebolla picada en cuadrit...","[' 1 El frejol se adereza friendo el tocino, una vez dorado se pone la cebolla picada en cuadrit..."
6,19369,ingredientes,"[' 4 ajÃ­es amarillos', ' 1 cebolla mediana cortada en trozos medianos', ' 2 dientes ajos chicos...","[' 4 ajíes amarillos', ' 1 cebolla mediana cortada en trozos medianos', ' 2 dientes ajos chicos'..."
7,19369,pasos,"[' 1 Quitar las pepas y venas de los ajÃ­es amarillos, cortarlos en trozos medianos. Cortar la c...","[' 1 Quitar las pepas y venas de los ajíes amarillos, cortarlos en trozos medianos. Cortar la ce..."
8,19370,ingredientes,"[' Pollo', ' Sal, pimienta, comino, vinagre, orÃ©gano, sillao, mostaza o naranja', ' Maggy de po...","[' Pollo', ' Sal, pimienta, comino, vinagre, orégano, sillao, mostaza o naranja', ' Maggy de pol..."
9,19370,pasos,"[' 1 Primero salpimentamos el pollo con Sal, pimienta, comino, vinagre, orÃ©gano, sillao, mostaz...","[' 1 Primero salpimentamos el pollo con Sal, pimienta, comino, vinagre, orégano, sillao, mostaza..."


In [9]:
pd.Series({
    'recetas_con_senales_antes': int(codificacion_antes.sum()),
    'recetas_con_senales_despues': int(codificacion_despues.sum()),
    'campos_reparados': len(cambios_codificacion),
}, name='codificacion')

recetas_con_senales_antes       713
recetas_con_senales_despues       1
campos_reparados               1332
Name: codificacion, dtype: int64

## 4. País y tipo de comida

In [10]:
df['pais'].value_counts(dropna=False).head(15)

pais
España                  9486
Internacional           4436
Mexico                  1922
Peru                    1279
Perú                     720
Argentina                632
NaN                      496
Chile                    251
Colombia                 245
Venezuela                205
Puerto rico               86
Cuba                      74
Ecuador                   56
República dominicana      38
El salvador               37
Name: count, dtype: int64

In [11]:
df['pais'] = df['pais'].replace({
    'Peru': 'Perú', 'Mexico': 'México', 'Panama': 'Panamá',
    'Puerto rico': 'Puerto Rico', 'República dominicana': 'República Dominicana',
    'El salvador': 'El Salvador', 'Costa rica': 'Costa Rica',
})
df['ocasion'] = df['tipo_comida'].where(df['tipo_comida'].eq('Cumpleaños'))
df['tipo_comida'] = df['tipo_comida'].mask(df['tipo_comida'].eq('Cumpleaños'))

pd.crosstab(df['pais'], df['tipo_comida']).head(10)

tipo_comida,Acompañamiento,Cena,Desayuno,Entrante,Merienda,Plato principal,Postre
pais,,,,,,,
Argentina,24,0,10,35,71,210,91
Bolivia,0,0,0,0,0,8,0
Chile,9,2,7,34,4,53,27
Colombia,14,1,17,38,6,56,33
Costa Rica,0,0,2,1,0,0,3
Cuba,2,0,0,6,0,15,3
Ecuador,2,0,4,6,2,12,4
El Salvador,3,0,2,2,1,5,0
España,332,227,205,1547,200,2256,1130


## 5. Recuperar raciones

`Porciones` repite ingredientes en buena parte del archivo. Usamos `Comensales` cuando está disponible y mantenemos los rangos.

In [12]:
porciones_repetidas = df['porciones'].eq(df['ingredientes']).fillna(False) & df['porciones'].notna()
df.loc[porciones_repetidas, 'porciones'] = pd.NA

df.loc[porciones_repetidas, ['nombre', 'ingredientes', 'comensales']].head(5)

,nombre,ingredientes,comensales
id,,,
970,Tarta de pollo y zapallo,"['2 discos de masa de tarta', '1 pata y muslo o media suprema de pollo', '½ zapallo japonés', '1...",2 comensales
971,arepa de queso,"['masa', '2 tazas de agua (480 mililitros)', '2 tazas de harina precocida de maiz', '1 cucharada...",6 comensales
972,tarta de pollo y verduras,"['1 disco de masa de tarta', '1 suprema de pollo', '1 rama de puerro', '1 zucchini', '1 zanahori...",3 comensales
973,ceviche con chicharron de pota,"['800 gramos de filete de pescado', '1 kilogramo de limón', '1 ají limo', '1 cebolla roja', '1 r...",4 comensales
974,bombas de papas al horno,"['2 papas grandes', '1 huevo', '½ taza de pan rallado', '1 cucharada postre de nuez moscada', '½...",2 comensales


In [13]:
personas = pd.to_numeric(df['comensales'].str.extract(r'^(\d+)\s+comensal(?:es)?$', expand=False), errors='coerce')
porciones = pd.to_numeric(df['porciones'].where(df['porciones'].str.fullmatch(r'\d+(?:[.,]\d+)?').fillna(False))
                                      .str.replace(',', '.', regex=False), errors='coerce')
rangos = df['porciones'].str.extract(r'^(\d+)\s*(?:a|o|-|–|—)\s*(\d+)$').apply(pd.to_numeric, errors='coerce')
conflicto_raciones = personas.notna() & porciones.notna() & personas.ne(porciones)

df['raciones_min'] = personas.combine_first(porciones).combine_first(rangos[0]).astype('Float64')
df['raciones_max'] = personas.combine_first(porciones).combine_first(rangos[1]).astype('Float64')
df['raciones_unidad'] = pd.Series(pd.NA, index=df.index, dtype='string')
df.loc[df['raciones_min'].notna(), 'raciones_unidad'] = 'porciones'
df.loc[personas.notna(), 'raciones_unidad'] = 'personas'
df['raciones_fuente'] = pd.Series(pd.NA, index=df.index, dtype='string')
df.loc[df['raciones_min'].notna(), 'raciones_fuente'] = 'Porciones'
df.loc[personas.notna(), 'raciones_fuente'] = 'Comensales'

raciones_invalidas = conflicto_raciones | df['raciones_min'].le(0) | df['raciones_max'].lt(df['raciones_min'])
df.loc[raciones_invalidas.fillna(False), ['raciones_min', 'raciones_max']] = pd.NA

df[['nombre', 'comensales', 'porciones', 'raciones_min', 'raciones_max', 'raciones_unidad']].head(8)

,nombre,comensales,porciones,raciones_min,raciones_max,raciones_unidad
id,,,,,,
1,Berenjenas rellenas,NaN,4,4.0,4.0,porciones
2,Alcachofas al horno con pico de gallo,NaN,4,4.0,4.0,porciones
3,Arroz basmati salteado con Heura y verduras variadas al curry,NaN,4,4.0,4.0,porciones
4,Tataki de atún,NaN,4,4.0,4.0,porciones
5,Merluza al vapor a la gallega,NaN,4,4.0,4.0,porciones
6,Curry de tirabeques y espárragos con leche de coco y arroz basmati,NaN,4,4.0,4.0,porciones
7,Crema de puerros y guisantes,NaN,4,4.0,4.0,porciones
8,Guiso de bacalao con alubias y acelgas,NaN,4,4.0,4.0,porciones


In [14]:
df[['raciones_min', 'raciones_max']].describe(percentiles=[0.5, 0.95, 0.99])

,raciones_min,raciones_max
count,13245.0,13245.0
mean,4.410495,4.414345
std,3.204628,3.206712
min,1.0,1.0
50%,4.0,4.0
95%,10.0,10.0
99%,15.0,15.0
max,100.0,100.0


## 6. Duración, valoración y votos

Los formatos incompletos permanecen como nulos; no se completan con promedios.

In [15]:
df['duracion'].value_counts(dropna=False).head(12)

duracion
NaN       6833
30m       3641
45m       3362
15m       2093
1h 30m    1955
10m        571
2h 30m     396
4h         194
24h        157
3h         136
30 m       114
1 h        106
Name: count, dtype: int64

In [16]:
duracion = df['duracion'].str.lower().str.extract(
    r'^(?:(?P<horas>\d+)\s*(?:horas?|hrs?|h))?\s*(?:(?P<minutos>\d+)\s*(?:minutos?|mins?|m))?$'
).apply(pd.to_numeric, errors='coerce')
df['duracion_minutos'] = (duracion['horas'].fillna(0).mul(60) + duracion['minutos'].fillna(0))
df['duracion_minutos'] = df['duracion_minutos'].where(duracion.notna().any(axis=1) & df['duracion_minutos'].gt(0)).astype('Int64')

valoracion = df['valoracion_y_votos'].str.extract(r'Valoración:\s*(?P<puntuacion>\d+(?:[.,]\d+)?)\s*\((?P<votos>\d+)')
df['valoracion'] = pd.to_numeric(valoracion['puntuacion'].str.replace(',', '.', regex=False), errors='coerce').astype('Float64')
df['votos'] = pd.to_numeric(valoracion['votos'], errors='coerce').astype('Int64')

df[['duracion', 'duracion_minutos', 'valoracion', 'votos']].head(8)

,duracion,duracion_minutos,valoracion,votos
id,,,,
1,45 min,45,<NA>,<NA>
2,60 min,60,<NA>,<NA>
3,40 min,40,<NA>,<NA>
4,60 min,60,<NA>,<NA>
5,45 min,45,<NA>,<NA>
6,40 min,40,<NA>,<NA>
7,45 min,45,<NA>,<NA>
8,60 min,60,<NA>,<NA>


In [17]:
df.loc[df['duracion'].notna() & df['duracion_minutos'].isna(), ['nombre', 'duracion']].head(12)

,nombre,duracion
id,,
19381,queque de naranja en olla,1h15
19387,queque de chocolate en sarten,1h 15
19424,arroz con pato para restaurante,20mi
19592,crema de olluco con curcuma,25 ms
19593,albondigas veganas con salsa roja,30 ms
19597,carapulcra,1 h m
19621,arroz chaufa peruano,1 h m
19677,sopa seca,45ms
19752,causa de pulpo al olivo,1 30 m


## 7. Convertir listas y retirar campos redundantes

Las comas dentro de una descripción no delimitan ingredientes. Solo interpretamos las listas ya presentes en el archivo.

In [18]:
def leer_lista(texto):
    if pd.isna(texto) or not texto.startswith('[') or len(texto) > 50000:
        return pd.NA
    try:
        lista = ast.literal_eval(texto)
    except (ValueError, SyntaxError, RecursionError):
        return pd.NA
    return lista if isinstance(lista, list) and all(isinstance(x, str) for x in lista) else pd.NA

df['ingredientes_lista'] = df['ingredientes'].map(leer_lista)
df['pasos_lista'] = df['pasos'].map(leer_lista)
df['categoria_lista'] = df['categoria'].map(leer_lista)
df['n_ingredientes'] = df['ingredientes_lista'].str.len().astype('Int64')
df['n_pasos'] = df['pasos_lista'].str.len().astype('Int64')

df[['nombre', 'n_ingredientes', 'n_pasos']].head(10)

,nombre,n_ingredientes,n_pasos
id,,,
1,Berenjenas rellenas,<NA>,<NA>
2,Alcachofas al horno con pico de gallo,<NA>,<NA>
3,Arroz basmati salteado con Heura y verduras variadas al curry,<NA>,<NA>
4,Tataki de atún,<NA>,<NA>
5,Merluza al vapor a la gallega,<NA>,<NA>
6,Curry de tirabeques y espárragos con leche de coco y arroz basmati,<NA>,<NA>
7,Crema de puerros y guisantes,<NA>,<NA>
8,Guiso de bacalao con alubias y acelgas,<NA>,<NA>
9,Bacalao con habas y espárragos,<NA>,<NA>


In [19]:
nombre_comparacion = df[['nombre', 'nombre_auxiliar']].apply(
    lambda s: s.str.casefold().str.normalize('NFKD').str.replace(r'[\u0300-\u036f]', '', regex=True)
)
auxiliar_distinto = nombre_comparacion['nombre_auxiliar'].notna() & nombre_comparacion['nombre'].ne(nombre_comparacion['nombre_auxiliar'])

df.loc[auxiliar_distinto, ['nombre', 'nombre_auxiliar']].head(8)

,nombre,nombre_auxiliar
id,,
970,Tarta de pollo y zapallo,tarta de pollo y calabaza
1075,6 aperitivos fáciles para Nochevieja,https://www.recetasgratis.net/articulo aperitivos faciles para nochevieja
1282,galletas saladas de navidad,https://www.recetasgratis.net/video receta de galletas saladas de navidad
1285,arbol navideno de queso y frutas,https://www.recetasgratis.net/video receta de arbol navideno de queso y frutas
1287,renos navidenos para ninos,https://www.recetasgratis.net/video receta de renos navidenos para ninos
1300,aperitivos vegetarianos para sorprender,https://www.recetasgratis.net/articulo aperitivos vegetarianos para sorprender
1309,aperitivos frios y faciles para fiestas especial navidad,https://www.recetasgratis.net/articulo aperitivos frios y faciles para fiestas especial navidad
1336,dorinachos preparados,https://www.recetasgratis.net/video receta de dorinachos preparados


In [20]:
columnas_retiradas = pd.DataFrame({
    'campo': [*columnas_vacias, 'nombre_auxiliar'],
    'motivo': ['completamente_vacio'] * len(columnas_vacias) + ['nombre_auxiliar_inconsistente'],
})
df = df.drop(columns='nombre_auxiliar')

columnas_retiradas

,campo,motivo
0,calorias,completamente_vacio
1,unnamed_16,completamente_vacio
2,unnamed_17,completamente_vacio
3,nombre_auxiliar,nombre_auxiliar_inconsistente


## 8. Registros incompletos y duplicados

Retiramos registros sin nombre, ingredientes o pasos, y copias del mismo contenido. Guardamos sus IDs y el motivo.

In [21]:
claves = df[['nombre', 'ingredientes', 'pasos']].apply(lambda s: s.str.casefold().str.normalize('NFKC'))
duplicados = claves.duplicated(keep=False) & claves.notna().all(axis=1)

df.loc[duplicados, ['id', 'nombre', 'fuente']].sort_values('nombre').head(12)

,id,nombre,fuente
id,,,
801,801,AJIACO DE CAIGUAS,www.yanuq.com
544,544,AJIACO DE CAIGUAS,www.yanuq.com
800,800,AJIACO DE PAPAS,www.yanuq.com
542,542,AJIACO DE PAPAS,www.yanuq.com
816,816,ANTICUCHOS DE ATUN,www.yanuq.com
243,243,ANTICUCHOS DE ATUN,www.yanuq.com
539,539,ARRIMADO DE COL,www.yanuq.com
799,799,ARRIMADO DE COL,www.yanuq.com
857,857,BATIDO DE FRESAS Y PLÁTANO,www.yanuq.com


In [22]:
claves_completas = claves.dropna()
indices_unicos = claves_completas.drop_duplicates().index
limpias = df.loc[indices_unicos].copy()

grupos = claves_completas.groupby(list(claves_completas.columns), sort=False).ngroup()
id_referencia = df.loc[claves_completas.index, 'id'].groupby(grupos, sort=False).transform('first')
descartadas = df.loc[~df.index.isin(indices_unicos), ['id', 'nombre', 'url', 'fuente']].copy()
descartadas['motivo'] = 'contenido_incompleto'
es_copia = descartadas.index.isin(claves_completas.index)
descartadas.loc[es_copia, 'motivo'] = 'duplicado_de_contenido'
descartadas['conservada_como_id'] = id_referencia.reindex(descartadas.index).where(es_copia)

descartadas.groupby('motivo').size().rename('registros')

motivo
contenido_incompleto      212
duplicado_de_contenido     40
Name: registros, dtype: int64

## 9. Una fila por ingrediente

In [23]:
ingredientes = (
    limpias[['id', 'ingredientes_lista']].dropna(subset=['ingredientes_lista'])
           .explode('ingredientes_lista').rename(columns={'id': 'receta_id', 'ingredientes_lista': 'texto'})
           .reset_index(drop=True)
)
ingredientes['texto'] = ingredientes['texto'].astype('string').str.normalize('NFC').str.strip().str.replace(r'\s+', ' ', regex=True)
ingredientes = ingredientes.replace(r'^\s*$', pd.NA, regex=True).dropna(subset=['texto'])
ingredientes['posicion'] = ingredientes.groupby('receta_id', sort=False).cumcount().add(1)

ingredientes.head(12)

,receta_id,texto,posicion
0,970,2 discos de masa de tarta,1
1,970,1 pata y muslo o media suprema de pollo,2
2,970,½ zapallo japonés,3
3,970,1 huevo,4
4,970,60 gramos de queso fresco,5
5,970,1 cucharada postre de cúrcuma,6
6,970,1 cucharada postre de mostaza,7
7,970,½ cucharada postre de nuez moscada,8
8,971,masa,1
9,971,2 tazas de agua (480 mililitros),2


## 10. Extraer cantidades y unidades explícitas

Separamos fracciones, rangos, unidad y descripción. Convertimos kg a g y litros a ml; tazas, cucharadas y piezas mantienen su unidad.

In [24]:
texto_medidas = (ingredientes['texto'].str.lower()
                 .str.replace(r'(\d)([¼½¾⅐⅑⅒⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞])', r'\1 \2', regex=True)
                 .str.normalize('NFKC').str.replace('⁄', '/', regex=False))

numero = r'(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:[.,]\d+)?)'
unidad = r'(?:kilogramos?|kilos?|kg|gramos?|grs?|g|mililitros?|ml|litros?|lt|l|tazas?|cucharadas?\s+soperas?|cucharadas?\s+postre|cucharaditas?|cucharadas?|cdas?|cdtas?|pizcas?|unidades?|piezas?|dientes?|filetes?|latas?|paquetes?|manojos?|chorros?)'
extraidas = texto_medidas.str.extract(
    rf'^(?P<cantidad_min_texto>{numero})(?:\s*(?:a|o|-|–)\s*(?P<cantidad_max_texto>{numero}))?\s*'
    rf'(?:(?P<unidad_texto>{unidad})\b\.?)?\s*(?P<descripcion>.*)$'
)

ingredientes = ingredientes.join(extraidas)
for extremo in ['min', 'max']:
    valores = ingredientes[f'cantidad_{extremo}_texto']
    fracciones = valores.str.extract(r'^(?:(?P<entero>\d+)\s+)?(?P<numerador>\d+)/(?P<denominador>\d+)$').apply(pd.to_numeric, errors='coerce')
    fraccion = fracciones['entero'].fillna(0) + fracciones['numerador'].div(fracciones['denominador'].where(fracciones['denominador'].gt(0)))
    ingredientes[f'cantidad_{extremo}'] = pd.to_numeric(valores.str.replace(',', '.', regex=False), errors='coerce').astype('Float64').fillna(fraccion)

ingredientes['cantidad_max'] = ingredientes['cantidad_max'].fillna(ingredientes['cantidad_min'])
ingredientes['cantidad_anomala'] = (ingredientes['cantidad_min'].le(0) | ingredientes['cantidad_max'].lt(ingredientes['cantidad_min'])).fillna(False)
ingredientes.loc[ingredientes['cantidad_anomala'], ['cantidad_min', 'cantidad_max']] = pd.NA
ingredientes['descripcion'] = ingredientes['descripcion'].fillna(texto_medidas).str.replace(r'^de\s+', '', regex=True).str.strip()

ingredientes[['texto', 'cantidad_min', 'cantidad_max', 'unidad_texto', 'descripcion']].head(15)

,texto,cantidad_min,cantidad_max,unidad_texto,descripcion
0,2 discos de masa de tarta,2.0,2.0,<NA>,discos de masa de tarta
1,1 pata y muslo o media suprema de pollo,1.0,1.0,<NA>,pata y muslo o media suprema de pollo
2,½ zapallo japonés,0.5,0.5,<NA>,zapallo japonés
3,1 huevo,1.0,1.0,<NA>,huevo
4,60 gramos de queso fresco,60.0,60.0,gramos,queso fresco
5,1 cucharada postre de cúrcuma,1.0,1.0,cucharada postre,cúrcuma
6,1 cucharada postre de mostaza,1.0,1.0,cucharada postre,mostaza
7,½ cucharada postre de nuez moscada,0.5,0.5,cucharada postre,nuez moscada
8,masa,<NA>,<NA>,<NA>,masa
9,2 tazas de agua (480 mililitros),2.0,2.0,tazas,agua (480 mililitros)


In [25]:
equivalencias = pd.DataFrame([
    ('kg', 'g', 1000), ('kilogramo', 'g', 1000), ('kilogramos', 'g', 1000),
    ('kilo', 'g', 1000), ('kilos', 'g', 1000),
    ('g', 'g', 1), ('gr', 'g', 1), ('grs', 'g', 1), ('gramo', 'g', 1), ('gramos', 'g', 1),
    ('l', 'ml', 1000), ('lt', 'ml', 1000), ('litro', 'ml', 1000), ('litros', 'ml', 1000),
    ('ml', 'ml', 1), ('mililitro', 'ml', 1), ('mililitros', 'ml', 1),
], columns=['unidad_texto', 'unidad_base', 'factor'])

ingredientes = ingredientes.merge(equivalencias, on='unidad_texto', how='left', validate='many_to_one')
ingredientes['cantidad_base_min'] = ingredientes['cantidad_min'].mul(ingredientes['factor']).astype('Float64')
ingredientes['cantidad_base_max'] = ingredientes['cantidad_max'].mul(ingredientes['factor']).astype('Float64')
ingredientes['es_posible_encabezado'] = texto_medidas.str.fullmatch(r'(?:para (?:la|el) )?(?:masa|relleno|salsa|cobertura|decoracion):?').fillna(False).to_numpy()

ingredientes.loc[ingredientes['cantidad_base_min'].notna(), ['texto', 'cantidad_base_min', 'unidad_base']].head(12)

,texto,cantidad_base_min,unidad_base
4,60 gramos de queso fresco,60.0,g
12,250 gramos de queso semi blando fresco,250.0,g
13,100 gramos de queso duro fresco,100.0,g
14,125 mililitros de agua adicional,125.0,ml
16,350 gramos de queso para fundir (opcional),350.0,g
27,800 gramos de filete de pescado,800.0,g
28,1 kilogramo de limón,1000.0,g
32,500 gramos de pota,500.0,g
34,300 gramos de harina sin preparar,300.0,g
35,150 gramos de chuño,150.0,g


In [26]:
calidad_ingredientes = (
    ingredientes.assign(con_cantidad=ingredientes['cantidad_min'].notna(),
                        con_medida_base=ingredientes['cantidad_base_min'].notna())
                .groupby('receta_id').agg(lineas=('texto', 'size'),
                                         con_cantidad=('con_cantidad', 'sum'),
                                         con_medida_base=('con_medida_base', 'sum'))
                .assign(cobertura_cantidad_pct=lambda t: t['con_cantidad'].div(t['lineas']).mul(100).round(1))
)
calidad_ingredientes.head(10)

,lineas,con_cantidad,con_medida_base,cobertura_cantidad_pct
receta_id,,,,
1000,20,18,2,90.0
10000,10,9,1,90.0
10001,9,9,2,100.0
10002,9,9,4,100.0
10003,9,9,3,100.0
10004,12,12,0,100.0
10005,6,6,0,100.0
10006,9,9,0,100.0
10007,7,7,3,100.0


## 11. Casos pendientes

Los ingredientes sin unidad o sin cantidad quedan visibles. Las etiquetas nutricionales se mantienen como datos de la fuente.

In [27]:
etiquetas = limpias['categoria_2'].fillna('').str.get_dummies(sep=', ')
problemas = pd.DataFrame({
    'raciones_ausentes': limpias['raciones_min'].isna(),
    'raciones_en_intervalo': limpias['raciones_min'].ne(limpias['raciones_max']),
    'confirmar_porciones_como_personas': limpias['raciones_unidad'].eq('porciones'),
    'duracion_ausente_o_ambigua': limpias['duracion_minutos'].isna(),
    'ingredientes_sin_lista': limpias['ingredientes_lista'].isna(),
    'tipo_comida_ausente': limpias['tipo_comida'].isna(),
    'codificacion_pendiente': codificacion_despues.reindex(limpias.index),
    'etiquetas_contradictorias': etiquetas['Sin grasa'].eq(1) & etiquetas['Alto en grasas'].eq(1),
}).fillna(False)
problemas = problemas.join(calidad_ingredientes[['cobertura_cantidad_pct']])
problemas['cantidades_incompletas'] = problemas['cobertura_cantidad_pct'].lt(100) | problemas['cobertura_cantidad_pct'].isna()

motivos = (problemas.drop(columns='cobertura_cantidad_pct').rename_axis('id').reset_index()
                    .melt(id_vars='id', var_name='motivo', value_name='pendiente')
                    .loc[lambda t: t['pendiente']].drop(columns='pendiente'))
revision = limpias.join(motivos.groupby('id')['motivo'].agg('; '.join).rename('motivos_revision'), how='inner')

motivos['motivo'].value_counts().rename('recetas')

motivo
tipo_comida_ausente                  9326
cantidades_incompletas               8733
duracion_ausente_o_ambigua           6594
raciones_ausentes                    6588
ingredientes_sin_lista                929
confirmar_porciones_como_personas     667
etiquetas_contradictorias             391
raciones_en_intervalo                  35
codificacion_pendiente                  1
Name: recetas, dtype: int64

In [28]:
ingredientes.loc[ingredientes['cantidad_min'].isna() | ingredientes['unidad_texto'].isna(),
                 ['receta_id', 'texto', 'cantidad_min', 'unidad_texto']].head(15)

,receta_id,texto,cantidad_min,unidad_texto
0,970,2 discos de masa de tarta,2.0,<NA>
1,970,1 pata y muslo o media suprema de pollo,1.0,<NA>
2,970,½ zapallo japonés,0.5,<NA>
3,970,1 huevo,1.0,<NA>
8,971,masa,<NA>,<NA>
15,971,relleno,<NA>,<NA>
17,972,1 disco de masa de tarta,1.0,<NA>
18,972,1 suprema de pollo,1.0,<NA>
19,972,1 rama de puerro,1.0,<NA>
20,972,1 zucchini,1.0,<NA>


## 12. Comparar antes y después

In [29]:
comparacion = pd.DataFrame({
    'antes': [len(raw), int(raw.isna().all().sum()),
              int(pd.to_numeric(raw['Porciones'], errors='coerce').notna().sum()),
              int(raw[['Nombre', 'Ingredientes', 'Pasos']].replace('[]', pd.NA).isna().any(axis=1).sum()),
              int(codificacion_antes.sum()), int(duplicados.sum())],
    'despues': [len(limpias), int(limpias.isna().all().sum()),
                int(limpias['raciones_min'].notna().sum()),
                int(limpias[['nombre', 'ingredientes', 'pasos']].isna().any(axis=1).sum()),
                int(codificacion_despues.reindex(limpias.index).sum()), 0],
}, index=['registros', 'columnas_totalmente_vacias', 'raciones_en_campos_numericos',
          'registros_sin_contenido_completo', 'recetas_con_senales_de_codificacion', 'registros_en_grupos_duplicados'])

comparacion

,antes,despues
registros,20085,19833
columnas_totalmente_vacias,3,0
raciones_en_campos_numericos,632,13245
registros_sin_contenido_completo,212,0
recetas_con_senales_de_codificacion,713,1
registros_en_grupos_duplicados,80,0


In [30]:
limpias = limpias.drop(columns=['porciones', 'comensales']).rename(columns={
    'duracion': 'duracion_fuente', 'valoracion_y_votos': 'valoracion_y_votos_fuente',
    'categoria_2': 'etiquetas_nutricionales_fuente',
})
limpias[['pais', 'tipo_comida', 'dificultad', 'raciones_unidad']] = limpias[['pais', 'tipo_comida', 'dificultad', 'raciones_unidad']].astype('category')
limpias.info()

<class 'pandas.DataFrame'>
Index: 19833 entries, 1 to 20085
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   id                              19833 non-null  string  
 1   nombre                          19833 non-null  str     
 2   url                             19833 non-null  string  
 3   ingredientes                    19833 non-null  str     
 4   pasos                           19833 non-null  str     
 5   pais                            19339 non-null  category
 6   duracion_fuente                 13252 non-null  str     
 7   categoria                       7512 non-null   str     
 8   contexto                        11513 non-null  str     
 9   valoracion_y_votos_fuente       13320 non-null  str     
 10  tipo_comida                     10507 non-null  category
 11  dificultad                      13782 non-null  category
 12  etiquetas_nutricionales_fuente  88

## 13. Exportar

In [31]:
assert len(raw) == len(limpias) + len(descartadas)
assert limpias['id'].is_unique
assert not claves.loc[limpias.index].duplicated().any()
assert ingredientes['receta_id'].isin(limpias['id']).all()
assert hashlib.sha256(ORIGEN.read_bytes()).hexdigest() == hash_original

PROCESADOS.mkdir(parents=True, exist_ok=True)
REPORTES.mkdir(parents=True, exist_ok=True)

salida_recetas = limpias.copy()
salida_revision = revision.copy()
for tabla in [salida_recetas, salida_revision]:
    for campo in ['ingredientes_lista', 'pasos_lista', 'categoria_lista']:
        tabla[campo] = tabla[campo].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else pd.NA)

salidas = {
    PROCESADOS / 'recetas_limpias.csv': salida_recetas,
    PROCESADOS / 'ingredientes_lineas.csv': ingredientes,
    PROCESADOS / 'recetas_revision.csv': salida_revision,
    REPORTES / 'recetas_descartadas.csv': descartadas,
    REPORTES / 'cambios_texto.csv': cambios_codificacion,
    REPORTES / 'comparacion_limpieza.csv': comparacion.rename_axis('metrica').reset_index(),
    REPORTES / 'perfil_fuentes.csv': perfil_fuentes.reset_index(),
}
for ruta, tabla in salidas.items():
    tabla.to_csv(ruta, index=False, encoding='utf-8', lineterminator='\n')

resumen = {
    'sha256_original': hash_original,
    'registros_originales': len(raw), 'recetas_limpias': len(limpias),
    'registros_descartados': len(descartadas), 'recetas_para_revision': len(revision),
    'lineas_ingredientes': len(ingredientes),
    'raciones_recuperadas': int(limpias['raciones_min'].notna().sum()),
    'campos_codificacion_reparados': len(cambios_codificacion),
    'motivos_revision': {str(k): int(v) for k, v in motivos['motivo'].value_counts().items()},
}
(REPORTES / 'resumen_calidad.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

pd.DataFrame([{'archivo': ruta.name, 'filas': len(tabla)} for ruta, tabla in salidas.items()])

,archivo,filas
0,recetas_limpias.csv,19833
1,ingredientes_lineas.csv,161885
2,recetas_revision.csv,11693
3,recetas_descartadas.csv,252
4,cambios_texto.csv,1332
5,comparacion_limpieza.csv,6
6,perfil_fuentes.csv,6


Siguiente análisis: resolver líneas ambiguas, comprobar porciones y acompañamientos y asociar ingredientes con el catálogo comercial.